This Notebook uses constructed data offline. Fixed filing facts and reference figures are historical teaching materials; new calculations do not reproduce a market strategy.

[Entry modes and preparation](../NOTEBOOK_GUIDE.md). See this guide for run steps, data requirements, and expected outputs.

In [ ]:
from pathlib import Path
import os, sys
from uuid import uuid4
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.chdir(ROOT)
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch12_statements_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Default offline teaching; current empirical French execution SKIPPED (permissions unresolved).")
print("Run directory:", RUN_DIR)


# Chapter 12: financial statements for quantitative research

Typed functions in `src/quant_course/statement_metrics.py` recalculate Apple FY2024/FY2023 filing rows and compare Spire FY2023 original/restated filings. Amounts come from the SEC documents linked in the chapter. PIT timestamps, issuer labels and the market-cap illustration are constructed; no historical prices, vendor ingestion logs or factor performance are loaded. Match currency, unit, period and accession first.

In [ ]:
from pathlib import Path
import math
import sys

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from quant_course.statement_metrics import (
    apple_facts,
    derive_accounting_factors,
    fcf_from_statement_lines,
    fcf_yield,
    point_in_time_revision_demo,
    revenue_growth,
    spire_cash_flow_facts,
)

print("工作目录：", ROOT)
print("单位约定：Apple USD millions；Spire USD thousands")


## Apple: margins, cash flow and book leverage

Both years are comparative columns in the same 2024 Form 10-K accession. Convert the negative CapEx cash-flow line to positive spending before computing FCF. Net debt includes commercial paper and term debt less cash and equivalents; it neither subtracts marketable securities nor adds leases. The code checks FCF of USD 108,807 million and USD 99,584 million, and a zero balance-sheet residual.

In [ ]:
apple_2024 = apple_facts(2024)
apple_2023 = apple_facts(2023)
metrics_2024 = derive_accounting_factors(apple_2024)
metrics_2023 = derive_accounting_factors(apple_2023)
revenue_growth_2024 = revenue_growth(
    apple_2024["revenue"],
    apple_2023["revenue"],
)

print({
    "FY2024_revenue_USD_millions": apple_2024["revenue"].value,
    "FY2024_FCF_USD_millions": metrics_2024["fcf"],
    "FY2024_FCF_margin": metrics_2024["fcf_margin"],
    "FY2024_operating_margin": metrics_2024["operating_margin"],
    "FY2024_EBITDA_proxy_USD_millions": metrics_2024["ebitda_proxy"],
    "FY2024_net_debt_USD_millions": metrics_2024["net_debt"],
    "FY2024_balance_sheet_residual": metrics_2024["balance_sheet_residual"],
    "FY2024_revenue_growth_vs_FY2023": revenue_growth_2024,
})
assert metrics_2024["fcf"] == 108_807.0
assert metrics_2023["fcf"] == 99_584.0
assert metrics_2024["balance_sheet_residual"] == 0.0
assert math.isclose(revenue_growth_2024, 391_035 / 383_285 - 1, abs_tol=1e-14)


## Spire: keep versions separate

Two accessions identify the original and restated filings. Convert the negative CapEx cash-flow line to positive spending, then compute CFO−CapEx. Both versions give −USD 53,659 thousand here, despite changes in CFO and CapEx. Unchanged FCF does not imply unchanged accounts.

In [ ]:
spire_rows = []
for revision in ("original", "restated"):
    cfo, capex_cash_flow = spire_cash_flow_facts(revision)
    fcf = fcf_from_statement_lines(cfo, capex_cash_flow)
    spire_rows.append({
        "revision": revision,
        "accession": cfo.accession,
        "CFO_USD_thousands": cfo.value,
        "CapEx_spend_USD_thousands": -capex_cash_flow.value,
        "FCF_USD_thousands": fcf,
    })
spire_versions = pd.DataFrame(spire_rows)
display(spire_versions)
assert spire_versions["accession"].nunique() == 2
assert spire_versions["CFO_USD_thousands"].tolist() == [-23_622.0, -36_307.0]
assert spire_versions["CapEx_spend_USD_thousands"].tolist() == [30_037.0, 17_352.0]
assert spire_versions["FCF_USD_thousands"].tolist() == [-53_659.0, -53_659.0]


## A constructed two-clock PIT example

UTC strings describe a fabricated timeline, not Apple or Spire release times. Both available_at and ingested_at must be no later than signal_time (≤). Rank eligible versions by source available_at; ingestion is an eligibility gate rather than the version-ordering clock.

In [ ]:
pit = point_in_time_revision_demo()
display(pit[["signal_time", "accession", "revision_status", "cfo_usd_millions"]])
assert pd.isna(pit.loc[0, "cfo_usd_millions"])
assert pit.loc[1, "revision_status"] == "original"
assert pit.loc[2, "revision_status"] == "original"
assert pit.loc[3, "revision_status"] == "restated"


## Constructed market capitalization demonstrates units

The filing does not supply this example's historical market capitalization. The explicit input of USD 4,000,000 million is constructed; it is neither Apple's observed market value nor a tradable FCF-yield estimate. The output is 108807/4000000.

In [ ]:
demo_yield = fcf_yield(
    metrics_2024["fcf"],
    4_000_000.0,
    currency="USD",
    statement_unit="USD millions",
    market_cap_currency="USD",
    market_cap_unit="USD millions",
    market_cap_label="constructed teaching value",
)
print(f"Constructed FCF yield: {demo_yield:.4%}")
assert math.isclose(demo_yield, 108_807 / 4_000_000, abs_tol=1e-14)


## Frozen figures

The two existing PNGs display SEC filing amounts. Spire's horizontal axis indexes accession versions. The Notebook displays them without regenerating or saving them.

In [ ]:
display(Image(filename=str(ROOT / "figures" / "a3" / "ch12_apple_cashflow.png")))
display(Image(filename=str(ROOT / "figures" / "a3" / "ch12_spire_revision.png")))


## Limits

The historical Apple press-release check established a publication date only. The SEC header's raw acceptance string is not converted to UTC here. Spire's 8-K report date, filing date and 10-K/A date retain distinct meanings. Historical filing versions have not been joined to real vendor ingestion logs, security mappings, exchange calendars and market prices to form a complete PIT panel. Arithmetic checks cannot support historical FCF-yield or backtest claims. Chinese setup print lines state the working directory and units: Apple in USD millions and Spire in USD thousands.